# garbage_collection
**Prerequisites:** memory_model, references, mutability

**Outcome:** After this notebook you will know exactly how CPython reclaims memory, why reference counting alone is not enough, how the cyclic garbage collector works, and how to control and observe it in production code.

## The Problem

In [ ]:
import gc
import sys

# create two objects that reference each other
class Node:
    def __init__(self, name):
        self.name = name
        self.peer = None

a = Node("producer")
b = Node("consumer")
a.peer = b
b.peer = a

print(sys.getrefcount(a))   # 3: a, b.peer, getrefcount arg
print(sys.getrefcount(b))   # 3: b, a.peer, getrefcount arg

del a
del b

# both names are gone — but are the objects freed?
# refcount for each dropped to 1, not 0 — the cycle keeps them alive
print("names deleted — objects may still be in memory")

## The Concept

CPython uses two mechanisms to reclaim memory. The first is reference counting: every object tracks how many references point to it and is freed immediately when that count reaches zero. This handles the vast majority of objects. The second is the cyclic garbage collector: a separate periodic scan that finds groups of objects that reference each other in a cycle but are not reachable from anywhere in the running program. These objects will never be freed by reference counting alone because their counts never reach zero. The cyclic GC exists specifically to collect them.

## Minimal Example

In [ ]:
import gc

gc.disable()   # turn off automatic collection so we can observe manually

class Tracked:
    def __del__(self):
        print(f"{self.__class__.__name__} collected")

# create a cycle
x = Tracked()
y = Tracked()
x.ref = y
y.ref = x

del x
del y
print("deleted names — no collection yet")

collected = gc.collect()   # manually trigger the cyclic collector
print(f"gc collected {collected} objects")

gc.enable()

## How Python Does It

The cyclic GC organises all tracked objects into three generations — 0, 1, and 2. New objects start in generation 0. Objects that survive a collection are promoted to the next generation. Generation 0 is collected most frequently, generation 2 least frequently. This generational approach is based on the empirical observation that most objects die young. The collector works by computing the number of internal references within a candidate set, subtracting them from the total reference count, and identifying objects whose adjusted count is zero — meaning nothing outside the cycle references them.

In [ ]:
import gc

# inspect the three generations
print("thresholds:", gc.get_threshold())   # (700, 10, 10) by default
print("counts    :", gc.get_count())       # objects in each generation

# generation 0 collected when its count exceeds threshold[0]
# generation 1 collected every threshold[1] gen-0 collections
# generation 2 collected every threshold[2] gen-1 collections

## Building Up

In [ ]:
import gc

# observe __del__ being called when refcount hits zero (no cycle)
class Service:
    def __init__(self, name):
        self.name = name
    def __del__(self):
        print(f"Service({self.name}) freed")

s = Service("api")
print("about to delete")
del s   # refcount hits zero — freed immediately
print("delete done")

In [ ]:
import gc

# __del__ is NOT called immediately when a cycle exists
class Service:
    def __init__(self, name):
        self.name = name
    def __del__(self):
        print(f"Service({self.name}) freed")

gc.disable()

a = Service("producer")
b = Service("consumer")
a.partner = b
b.partner = a

del a
del b
print("deleted — but __del__ not yet called")

gc.collect()
print("after gc.collect()")

gc.enable()

In [ ]:
import gc

# gc.get_objects() returns all objects tracked by the cyclic collector
before = len(gc.get_objects())

class Job:
    pass

jobs = [Job() for _ in range(100)]
after = len(gc.get_objects())

print(f"tracked objects before: {before}")
print(f"tracked objects after : {after}")
print(f"difference            : {after - before}")

In [ ]:
import gc

# gc.garbage contains uncollectable objects (rare in modern Python)
# in Python 3.4+ cycles with __del__ are collectable
# gc.garbage is mostly empty unless you create truly uncollectable structures

gc.collect()
print("uncollectable:", gc.garbage)   # should be [] in normal code

In [ ]:
import gc

# manually tuning collection thresholds for a batch processing system
# that creates many short-lived objects in tight loops

print("default thresholds:", gc.get_threshold())

# increase gen-0 threshold to collect less frequently during bulk load
gc.set_threshold(1000, 15, 15)
print("tuned thresholds  :", gc.get_threshold())

# restore defaults
gc.set_threshold(700, 10, 10)

In [ ]:
import gc

# gc.freeze() promotes all current objects to permanent generation
# useful before forking — avoids child processes collecting parent objects
# available in Python 3.7+

gc.collect()   # clean up first
gc.freeze()    # freeze all current objects
print("frozen counts:", gc.get_freeze_count())

# new objects created after freeze() are tracked normally
# frozen objects are never collected

## Where It Breaks

In [ ]:
import gc
import sys

# memory leak: long-lived container accumulates references silently
_registry = []

def register_job(job_id):
    record = {"id": job_id, "payload": list(range(10_000))}
    _registry.append(record)   # refcount never drops — memory grows forever

for i in range(50):
    register_job(f"job_{i}")

print(f"registry holds {len(_registry)} records")
print(f"approx size: {sys.getsizeof(_registry)} bytes for the list alone")

In [ ]:
# disabling gc in a long-running process without manual collection
import gc

gc.disable()   # fine for short scripts, dangerous for long-running services

# if your code creates any cycles while gc is disabled
# those objects accumulate indefinitely until gc.enable() or gc.collect()

gc.enable()    # always re-enable or call gc.collect() periodically

## The Fix

In [ ]:
import weakref

# fix: use weakref for caches and registries to avoid holding objects alive
_registry = {}

class Job:
    def __init__(self, job_id):
        self.job_id = job_id

def register_job(job):
    _registry[job.job_id] = weakref.ref(job)   # does not increment refcount

def lookup_job(job_id):
    ref = _registry.get(job_id)
    return ref() if ref is not None else None   # returns None if collected

j = Job("job_1")
register_job(j)
print(lookup_job("job_1"))   # <__main__.Job object>

del j
print(lookup_job("job_1"))   # None — job freed, registry did not hold it

In [ ]:
import gc

# fix: break cycles explicitly before deleting
class Node:
    def __init__(self, name):
        self.name = name
        self.peer = None
    def __del__(self):
        print(f"Node({self.name}) freed")

a = Node("producer")
b = Node("consumer")
a.peer = b
b.peer = a

# break the cycle before deleting
a.peer = None

del a   # refcount hits zero — freed immediately
del b   # refcount hits zero — freed immediately
print("both freed without needing cyclic GC")

## In a Real System

In [ ]:
import gc
import time

# a batch processor that disables gc during the hot loop
# and collects manually between batches

def process_batch(records):
    return [{**r, "processed": True} for r in records]

batches = [
    [{"id": f"job_{i}"} for i in range(j, j + 100)]
    for j in range(0, 500, 100)
]

gc.disable()   # avoid GC pauses mid-batch

results = []
for i, batch in enumerate(batches):
    results.extend(process_batch(batch))

    if i % 2 == 0:         # collect between batches, not during
        gc.collect()

gc.enable()
gc.collect()               # final sweep

print(f"processed {len(results)} records")

## Performance

Reference counting is effectively free — one integer increment or decrement per reference operation. The cyclic GC pause depends on how many tracked objects exist in the generation being collected. In latency-sensitive systems like web servers or stream processors, GC pauses are real and measurable. The standard technique is to call `gc.disable()` at startup, run `gc.collect()` manually at known safe points (between requests, between batches), and use `gc.freeze()` after the import phase to exclude stable long-lived objects from future scans.

## Anti-Pattern

In [ ]:
import gc

# anti-pattern: calling gc.collect() inside a tight loop
def process_jobs(job_ids):
    results = []
    for job_id in job_ids:
        result = {"id": job_id, "status": "done"}
        results.append(result)
        gc.collect()   # catastrophic — GC scan on every iteration
    return results

# correct: collect once after the loop, not inside it
def process_jobs(job_ids):
    results = []
    for job_id in job_ids:
        result = {"id": job_id, "status": "done"}
        results.append(result)
    gc.collect()       # one collection after all work is done
    return results

output = process_jobs([f"job_{i}" for i in range(10)])
print(f"processed {len(output)} jobs")

## Interview Signals

- Why is reference counting alone not enough to manage memory in Python?
- What are the three generations in CPython's garbage collector and why does the generational approach work?
- When would you call `gc.disable()` in production and what must you do to compensate?
- What does `gc.freeze()` do and when would you use it?

## Exercise

In [ ]:
import gc
import weakref

class JobCache:
    """
    A cache that stores job results but must NOT prevent jobs
    from being garbage collected when no other references exist.

    Bug: the current implementation holds strong references,
    so jobs are never freed even after all external references are deleted.
    Fix it using weakref so the cache does not extend job lifetime.
    """
    def __init__(self):
        self._store = {}

    def put(self, job_id, job):
        self._store[job_id] = job          # bug: strong reference

    def get(self, job_id):
        return self._store.get(job_id)     # bug: returns strong reference


class Job:
    def __init__(self, job_id):
        self.job_id = job_id


cache = JobCache()
j = Job("job_1")
cache.put("job_1", j)

assert cache.get("job_1") is j, "should return the job while it exists"

del j
gc.collect()

assert cache.get("job_1") is None, "cache should not hold job alive after deletion"
print("all assertions passed")